In [ ]:
# CELL 1 — Connect to Drive and install YOLO
from google.colab import drive
drive.mount('/content/drive')

!pip install -q ultralytics

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 6.3 MB/s eta 0:00:00


In [ ]:
# save once, e.g. /content/drive/MyDrive/Thesis/Experiments1/custom_modules.py
%%writefile /content/drive/MyDrive/Thesis/Experiments1/custom_modules.py
import torch
import torch.nn as nn
import ultralytics.nn.tasks as tasks

class WeightedConcat(nn.Module):
    def __init__(self, dimension=1, n=2):
        super().__init__()
        self.d = dimension
        self.w = nn.Parameter(torch.ones(n), requires_grad=True)
        self.epsilon = 1e-4

    def forward(self, x):
        w = torch.relu(self.w)
        w = w / (w.sum() + self.epsilon)
        x = [xi * wi for xi, wi in zip(x, w)]
        return torch.cat(x, self.d)

# FIXED: register by NAME only. Do NOT do `tasks.Concat = WeightedConcat`.
# That was a global monkey-patch — it silently replaces EVERY plain Concat
# layer for the rest of the session, including layers in OTHER architectures
# (like CBAM's neck) that are supposed to stay plain Concat. Any YAML that
# wants weighted fusion must now use the literal type name "WeightedConcat"
# explicitly (see the Weighted Fusion yaml below, updated accordingly).
# Plain "Concat" anywhere now always stays plain Concat, regardless of
# what's been imported in this session — this is the fix.
tasks.WeightedConcat = WeightedConcat
print("WeightedConcat registered by name. Concat itself was NOT touched.")

Overwriting /content/drive/MyDrive/Thesis/Experiments1/custom_modules.py


In [ ]:
%run /content/drive/MyDrive/Thesis/Experiments1/custom_modules.py

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


CBAM Experiment

In [ ]:
# CELL 2 — Add the CBAM code (this is the "modification" itself)
import torch
import torch.nn as nn
import ultralytics.nn.tasks as tasks

class ChannelAttention(nn.Module):
    def __init__(self, c1, r=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        hidden = max(c1 // r, 8)
        self.fc = nn.Sequential(
            nn.Conv2d(c1, hidden, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, c1, 1, bias=False),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        return x * self.sigmoid(avg_out + max_out)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        padding = 3 if kernel_size == 7 else 1
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x_cat = torch.cat([avg_out, max_out], dim=1)
        return x * self.sigmoid(self.conv(x_cat))

class CBAM(nn.Module):
    def __init__(self, c1, c2=None, r=16, kernel_size=7):
        super().__init__()
        self.channel_attention = ChannelAttention(c1, r)
        self.spatial_attention = SpatialAttention(kernel_size)

    def forward(self, x):
        x = self.channel_attention(x)
        x = self.spatial_attention(x)
        return x

tasks.CBAM = CBAM
print("Done — CBAM is ready to use.")

Done — CBAM is ready to use.


In [ ]:
# Check 1 — is your yaml uploaded?
!find "/content/drive/MyDrive/Thesis/Experiments1" -iname "*cbam*"

# Check 2 — where is your dataset's data.yaml?
!find "/content/drive/MyDrive/Thesis" -iname "data.yaml"

/content/drive/MyDrive/Thesis/Experiments1/yolo26n-cbam.yaml
/content/drive/MyDrive/Thesis/uppdms_baseline1/data.yaml


In [ ]:
%%writefile /content/drive/MyDrive/Thesis/Experiments1/yolo26n-cbam.yaml
nc: 8
end2end: True
reg_max: 1

scales:
  n: [0.50, 0.25, 1024]

backbone:
  - [-1, 1, Conv, [64, 3, 2]]                # 0  - P1/2
  - [-1, 1, Conv, [128, 3, 2]]                # 1  - P2/4
  - [-1, 2, C3k2, [256, False, 0.25]]         # 2
  - [-1, 1, Conv, [256, 3, 2]]                # 3  - P3/8
  - [-1, 2, C3k2, [512, False, 0.25]]         # 4
  - [-1, 1, CBAM, [128]]                      # 5
  - [-1, 1, Conv, [512, 3, 2]]                # 6  - P4/16
  - [-1, 2, C3k2, [512, True]]                # 7
  - [-1, 1, CBAM, [128]]                      # 8
  - [-1, 1, Conv, [1024, 3, 2]]               # 9  - P5/32
  - [-1, 2, C3k2, [1024, True]]               # 10
  - [-1, 1, SPPF, [1024, 5, 3, True]]         # 11
  - [-1, 2, C2PSA, [1024]]                    # 12
  - [-1, 1, CBAM, [256]]                      # 13

head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]   # 14
  - [[-1, 8], 1, Concat, [1]]                     # 15
  - [-1, 2, C3k2, [512, True]]                    # 16

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]   # 17
  - [[-1, 5], 1, Concat, [1]]                     # 18
  - [-1, 2, C3k2, [256, True]]                    # 19

  - [-1, 1, Conv, [256, 3, 2]]                    # 20
  - [[-1, 16], 1, Concat, [1]]                    # 21
  - [-1, 2, C3k2, [512, True]]                    # 22

  - [-1, 1, Conv, [512, 3, 2]]                    # 23
  - [[-1, 13], 1, Concat, [1]]                    # 24
  - [-1, 2, C3k2, [1024, True]]                   # 25

  - [[19, 22, 25], 1, Detect, [nc]]               # 26

Overwriting /content/drive/MyDrive/Thesis/Experiments1/yolo26n-cbam.yaml


In [ ]:
!find "/content/drive/MyDrive/Thesis/Experiments1" -iname "*cbam*"

/content/drive/MyDrive/Thesis/Experiments1/yolo26n-cbam.yaml


In [ ]:
# CELL 3 — Build the modified model, then train it
from ultralytics import YOLO

YAML_PATH = "/content/drive/MyDrive/Thesis/Experiments1/yolo26n-cbam.yaml"
DATA_PATH = "/content/drive/MyDrive/Thesis/uppdms_baseline1/data.yaml"

model = YOLO(YAML_PATH).load("yolo26n.pt")
model.info()

Transferred 98/717 items from pretrained weights
YOLO26n-cbam summary: 280 layers, 2,435,278 parameters, 2,435,278 gradients, 5.8 GFLOPs


(280, 2435278, 2435278, 5.781459776)

**Contamination check before (re)training CBAM:** if your existing `YOLO26_CBAM` results were produced by running the *old* version of the cell above (the one with `tasks.Concat = WeightedConcat`) in the same session as this training run, every plain `Concat` layer in `yolo26n-cbam.yaml` would have silently become `WeightedConcat` too — meaning the run measured CBAM + weighted fusion combined, not CBAM alone. With the fix above, re-running this cell now trains a clean CBAM-only model. Recommended: re-run this training cell once with the fixed registration and compare the new mAP50-95 against your existing `YOLO26_CBAM` result — if they match closely, contamination wasn't an issue in practice; if they differ meaningfully, use the new run in your thesis and note the correction.

In [ ]:
# CELL 5 — Full 100-epoch training run
from ultralytics import YOLO

model = YOLO(YAML_PATH).load("yolo26n.pt")   # fresh instance, same starting point as before

results = model.train(
    data=DATA_PATH,
    epochs=100,
    batch=16,
    imgsz=640,
    optimizer="AdamW",
    lr0=0.001,
    project="/content/drive/MyDrive/Thesis/Experiments1",
    name="YOLO26_CBAM",
)

Transferred 98/717 items from pretrained weights
Ultralytics 8.4.117 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Thesis/uppdms_baseline1/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/M

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: adaptive_max_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      1/100      2.62G      3.058      4.546    0.05975         74        640: 100% ━━━━━━━━━━━━ 84/84 2.8it/s 29.9s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 4.0it/s 2.2s
                   all        287       1239   0.000376     0.0714   0.000143   2.78e-05

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      2/100      3.22G      2.591      4.042    0.05387        128        640: 0% ──────────── 0/84  0.2s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: adaptive_max_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      2/100      3.22G      2.367      3.833    0.04333         73        640: 100% ━━━━━━━━━━━━ 84/84 3.0it/s 27.9s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 4.1it/s 2.2s
                   all        287       1239     0.0447      0.321     0.0114    0.00321

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      3/100      3.22G      2.232      3.736    0.04285        121        640: 0% ──────────── 0/84  0.2s

...

     98/100      3.22G      1.073     0.9278    0.02209         35        640: 100% ━━━━━━━━━━━━ 84/84 3.0it/s 28.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 3.7it/s 2.4s
                   all        287       1239      0.814      0.723      0.812      0.568

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     99/100      3.22G      0.994     0.7793    0.01976         72        640: 0% ──────────── 0/84  0.2s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: adaptive_max_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


     99/100      3.22G      1.052     0.9359    0.02108         33        640: 100% ━━━━━━━━━━━━ 84/84 3.1it/s 27.5s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 3.7it/s 2.4s
                   all        287       1239      0.811      0.735      0.811      0.567

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
    100/100      3.22G      1.283      1.065    0.02783         67        640: 0% ──────────── 0/84  0.2s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: adaptive_max_pool2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


    100/100      3.22G      1.063     0.9171    0.02138         37        640: 100% ━━━━━━━━━━━━ 84/84 3.1it/s 27.5s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 3.7it/s 2.4s
                   all        287       1239      0.784      0.747      0.811      0.567

100 epochs completed in 0.915 hours.
Optimizer stripped from /content/drive/MyDrive/Thesis/Experiments1/YOLO26_CBAM/weights/last.pt, 5.3MB
Optimizer stripped from /content/drive/MyDrive/Thesis/Experiments1/YOLO26_CBAM/weights/best.pt, 5.3MB

Validating /content/drive/MyDrive/Thesis/Experiments1/YOLO26_CBAM/weights/best.pt...
Ultralytics 8.4.117 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n-cbam summary (fused): 142 layers, 2,305,330 parameters, 0 gradients, 5.2 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 1.3it/s 6.8s
                   all        287       1239

In [ ]:
# Test the trained CBAM model on the test split
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/Thesis/Experiments1/YOLO26_CBAM/weights/best.pt")

test_results = model.val(
    data="/content/drive/MyDrive/Thesis/uppdms_baseline1/data.yaml",
    split="test",   # this is the key part — uses the test/ folder, not valid/
)

print("Test mAP50:", test_results.box.map50)
print("Test mAP50-95:", test_results.box.map)
print("Test Precision:", test_results.box.mp)
print("Test Recall:", test_results.box.mr)

Ultralytics 8.4.117 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n-cbam summary (fused): 142 layers, 2,305,330 parameters, 0 gradients, 5.2 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 1.6±2.1 ms, read: 0.8±1.8 MB/s, size: 16.9 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning /content/drive/MyDrive/Thesis/uppdms_baseline1/test/labels... 111 images, 0 backgrounds, 0 corrupt: 38% ━━━━╸─────── 111/287 3.2it/s 35.6s<54.2srequirements: Ultralytics requirement ['pi-heif'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 347ms
Prepared 1 package in 74ms
Installed 1 package in 2ms
 + pi-heif==1.4.0

requirements: AutoUpdate success ✅ 0.9s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

val: Scanning /content/drive/MyDrive/Thesis/uppdms_baseline1/test/labels... 286

In [ ]:
test_results = model.val(
    data="/content/drive/MyDrive/Thesis/uppdms_baseline1/data.yaml",
    split="test",
    project="/content/drive/MyDrive/Thesis/Experiments1",
    name="YOLO26_CBAM_test_results",
)

Ultralytics 8.4.117 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
WARNING ⚠️ val: Slow image access detected (ping: 0.7±0.5 ms, read: 24.1±8.2 MB/s, size: 26.7 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning /content/drive/MyDrive/Thesis/uppdms_baseline1/test/labels.cache... 286 images, 0 backgrounds, 1 corrupt: 100% ━━━━━━━━━━━━ 287/287 109.4Mit/s 0.0s
val: /content/drive/MyDrive/Thesis/uppdms_baseline1/test/images/image-1694-_jpg.rf.22bd19139662988a9829ed57dc011ecc.jpg: ignoring corrupt image/label: cannot identify image file '/content/drive/MyDrive/Thesis/uppdms_baseline1/test/images/image-1694-_jpg.rf.22bd19139662988a9829ed57dc011ecc.jpg'
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 2.3it/s 7.9s
                   all        286       1268       0.82      0.722      0.811       0.57
       

In [ ]:
# Test the original YOLO26 baseline on the same test set, for fair comparison
from ultralytics import YOLO

baseline_model = YOLO("/content/drive/MyDrive/Thesis/Experiments1/YOLO26_Baseline/weights/best.pt")

# FIXED: variable name had a stray space (`baseline_test_resu lts`), which
# caused a NameError on the print lines below even though the .val() call
# itself succeeded and saved its output folder correctly.
baseline_test_results = baseline_model.val(
    data="/content/drive/MyDrive/Thesis/uppdms_baseline1/data.yaml",
    split="test",
    project="/content/drive/MyDrive/Thesis/Experiments1",
    name="YOLO26_Baseline_test_results",
)

print("Baseline Test mAP50:", baseline_test_results.box.map50)
print("Baseline Test mAP50-95:", baseline_test_results.box.map)
print("Baseline Test Precision:", baseline_test_results.box.mp)
print("Baseline Test Recall:", baseline_test_results.box.mr)

Ultralytics 8.4.117 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 122 layers, 2,376,396 parameters, 0 gradients, 5.3 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 19.7±42.0 ms, read: 15.6±14.3 MB/s, size: 25.6 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning /content/drive/MyDrive/Thesis/uppdms_baseline1/test/labels.cache... 286 images, 0 backgrounds, 1 corrupt: 100% ━━━━━━━━━━━━ 287/287 92.6Mit/s 0.0s
val: /content/drive/MyDrive/Thesis/uppdms_baseline1/test/images/image-1694-_jpg.rf.22bd19139662988a9829ed57dc011ecc.jpg: ignoring corrupt image/label: cannot identify image file '/content/drive/MyDrive/Thesis/uppdms_baseline1/test/images/image-1694-_jpg.rf.22bd19139662988a9829ed57dc011ecc.jpg'
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.8it/s 4.8s
           

Weighted Fusion (All) Experiment

In [ ]:
yaml_content = """\
nc: 8
end2end: True
reg_max: 1

scales:
  n: [0.50, 0.25, 1024]

backbone:
  - [-1, 1, Conv, [64, 3, 2]]                # 0  - P1/2
  - [-1, 1, Conv, [128, 3, 2]]                # 1  - P2/4
  - [-1, 2, C3k2, [256, False, 0.25]]         # 2
  - [-1, 1, Conv, [256, 3, 2]]                # 3  - P3/8
  - [-1, 2, C3k2, [512, False, 0.25]]         # 4  - P3 output
  - [-1, 1, Conv, [512, 3, 2]]                # 5  - P4/16
  - [-1, 2, C3k2, [512, True]]                # 6  - P4 output
  - [-1, 1, Conv, [1024, 3, 2]]               # 7  - P5/32
  - [-1, 2, C3k2, [1024, True]]               # 8
  - [-1, 1, SPPF, [1024, 5, 3, True]]         # 9
  - [-1, 2, C2PSA, [1024]]                    # 10 - P5 output

head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]   # 11
  - [[-1, 6], 1, WeightedConcat, [1, 2]]          # 12 - weighted fusion (P5up + P4)
  - [-1, 2, C3k2, [512, True]]                    # 13

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]   # 14
  - [[-1, 4], 1, WeightedConcat, [1, 2]]          # 15 - weighted fusion (P4'up + P3)
  - [-1, 2, C3k2, [256, True]]                    # 16 - P3_out

  - [-1, 1, Conv, [256, 3, 2]]                    # 17
  - [[-1, 13], 1, WeightedConcat, [1, 2]]         # 18 - weighted fusion (P3'down + P4')
  - [-1, 2, C3k2, [512, True]]                    # 19 - P4_out

  - [-1, 1, Conv, [512, 3, 2]]                    # 20
  - [[-1, 10], 1, WeightedConcat, [1, 2]]         # 21 - weighted fusion (P4''down + P5)
  - [-1, 2, C3k2, [1024, True]]                   # 22 - P5_out

  - [[16, 19, 22], 1, Detect, [nc]]               # 23
"""
# FIXED: layer type changed from "Concat" to "WeightedConcat" explicitly.
# Previously this relied on the global `tasks.Concat = WeightedConcat`
# monkey-patch to work; now that registration is name-scoped only, the YAML
# must say WeightedConcat literally or these layers would silently fall back
# to being plain, unweighted Concat layers (i.e. this experiment would
# accidentally become a no-op — identical to the baseline architecture).

yaml_path = "/content/drive/MyDrive/Thesis/Experiments1/yolo26n-weighted-neck.yaml"
with open(yaml_path, "w") as f:
    f.write(yaml_content)

print("Saved:", yaml_path)

Saved: /content/drive/MyDrive/Thesis/Experiments1/yolo26n-weighted-neck.yaml


In [ ]:
from ultralytics import YOLO
import torch

model = YOLO(yaml_path)
model.info(verbose=False)

dummy = torch.zeros(1, 3, 640, 640)
with torch.no_grad():
    out = model.model(dummy)
print("Forward pass OK, output shapes:", [o.shape if hasattr(o, "shape") else type(o) for o in out])

Forward pass OK, output shapes: [<class 'str'>, <class 'str'>]


In [ ]:
print("Output type:", type(out))
print("Keys:", out.keys() if isinstance(out, dict) else "not a dict")

for k, v in out.items():
    if isinstance(v, (list, tuple)):
        print(f"{k}: list of {len(v)} tensors, shapes:", [t.shape for t in v])
    elif hasattr(v, "shape"):
        print(f"{k}: shape {v.shape}")
    else:
        print(f"{k}: {type(v)}")

Output type: <class 'dict'>
Keys: dict_keys(['one2many', 'one2one'])
one2many: <class 'dict'>
one2one: <class 'dict'>


In [ ]:
from ultralytics import YOLO
import torch

# Fresh model instance built from the modified yaml.
# Requires custom_modules.py to have been run already this session,
# otherwise Concat is still the plain, unpatched version.
model = YOLO(yaml_path)
model.info(verbose=False)  # sanity-check layer/param count

# Dummy forward pass (no real image, just checking shapes wire up
# correctly through all 4 WeightedConcat layers).
dummy = torch.zeros(1, 3, 640, 640)
with torch.no_grad():
    out = model.model(dummy)

# end2end: True models return a nested dict (one2many / one2one heads),
# not a flat list — so we walk it explicitly rather than assuming tensors.
print("Output type:", type(out))
for k, v in out.items():
    print(f"\n--- {k} ---")
    if isinstance(v, dict):
        for kk, vv in v.items():
            if isinstance(vv, (list, tuple)):
                print(f"  {kk}: list of {len(vv)}, shapes:",
                      [t.shape if hasattr(t, "shape") else type(t) for t in vv])
            elif hasattr(vv, "shape"):
                print(f"  {kk}: shape {vv.shape}")
            else:
                print(f"  {kk}: {type(vv)}")
    else:
        print(v)

Output type: <class 'dict'>

--- one2many ---
  boxes: shape torch.Size([1, 4, 8400])
  scores: shape torch.Size([1, 8, 8400])
  feats: list of 3, shapes: [torch.Size([1, 64, 80, 80]), torch.Size([1, 128, 40, 40]), torch.Size([1, 256, 20, 20])]

--- one2one ---
  boxes: shape torch.Size([1, 4, 8400])
  scores: shape torch.Size([1, 8, 8400])
  feats: list of 3, shapes: [torch.Size([1, 64, 80, 80]), torch.Size([1, 128, 40, 40]), torch.Size([1, 256, 20, 20])]


In [ ]:
"""
5-epoch sanity check for YOLO26n + WeightedConcat neck.
Purpose: confirm the model trains without errors and loss decreases —
NOT meant to produce usable metrics. Mirrors the CBAM sanity-check pattern.
"""

from ultralytics import YOLO

# Fresh instance — reusing a model object across .train() calls causes
# a KeyError: 'model' on the second call, so always instantiate new.
model = YOLO(yaml_path).load("yolo26n.pt")  # load pretrained weights where shapes match

results = model.train(
    data="/content/drive/MyDrive/Thesis/uppdms_baseline1/data.yaml",
    epochs=5,                # short trial only
    patience=100,            # disable early stopping for this short run
    batch=16,
    imgsz=640,
    project="/content/drive/MyDrive/Thesis/Experiments1",
    name="YOLO26_WeightedFusion_sanity_check",
    optimizer="AdamW",
    seed=42,
    deterministic=True,
    pretrained=True,
    # REMOVED: cls_remap=True was not a recognized Ultralytics train()
    # argument as of this codebase — verify against your installed
    # Ultralytics version's docs before re-adding it. Leaving it out
    # is safe; Ultralytics ignores/warns on unknown kwargs in most
    # versions but it's not guaranteed across versions.
    device="0",
)

Transferred 564/712 items from pretrained weights
Ultralytics 8.4.121 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Thesis/uppdms_baseline1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/My

In [ ]:
"""
Full training run: YOLO26n + WeightedConcat neck.
Hyperparameters copied from YOLO26_Baseline/args.yaml to keep this a
clean single-variable comparison (architecture is the only difference).
"""

from ultralytics import YOLO

# Fresh instance, as always — reusing model objects across .train() calls
# causes a KeyError: 'model'.
model = YOLO(yaml_path).load("yolo26n.pt")

results = model.train(
    data="/content/drive/MyDrive/Thesis/uppdms_baseline1/data.yaml",
    epochs=100,
    patience=20,              # matches baseline
    batch=16,
    imgsz=640,
    project="/content/drive/MyDrive/Thesis/Experiments1",
    name="YOLO26_WeightedFusion",
    optimizer="AdamW",
    lr0=0.001,                 # matches baseline (this run used 0.01 — fixed here)
    lrf=0.01,
    seed=42,
    deterministic=True,
    pretrained=True,
    # REMOVED: cls_remap=True was not a recognized Ultralytics train()
    # argument as of this codebase — verify against your installed
    # Ultralytics version's docs before re-adding it. Leaving it out
    # is safe; Ultralytics ignores/warns on unknown kwargs in most
    # versions but it's not guaranteed across versions.
    device="0",
)

Transferred 564/712 items from pretrained weights
Ultralytics 8.4.121 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Thesis/uppdms_baseline1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive

In [ ]:
model = YOLO("/content/drive/MyDrive/Thesis/Experiments1/YOLO26_WeightedFusion/weights/best.pt")
for name, module in model.model.named_modules():
    if module.__class__.__name__ == "WeightedConcat":
        w = torch.relu(module.w)
        w = w / (w.sum() + 1e-4)
        print(name, "normalized weights:", w.detach().cpu().numpy())

model.12 normalized weights: [    0.48446     0.51549]
model.15 normalized weights: [    0.49582     0.50413]
model.18 normalized weights: [    0.49106     0.50889]
model.21 normalized weights: [    0.58115     0.41879]


In [ ]:
from ultralytics import YOLO

YAML_PATH = "/content/drive/MyDrive/Thesis/uppdms_baseline1/data.yaml"

wf_best_path = "/content/drive/MyDrive/Thesis/Experiments1/YOLO26_WeightedFusion/weights/best.pt"
wf_best = YOLO(wf_best_path)

test_results_wf = wf_best.val(
    data=YAML_PATH,
    split="test",
    imgsz=640,
    batch=16,
    device=0,
    project="/content/drive/MyDrive/Thesis/Experiments1",
    name="YOLO26_WeightedFusion_test_results",
)

print("\nYOLO26 WEIGHTED FUSION - TEST RESULTS")
print("======================================")
print("Precision :", test_results_wf.box.mp)
print("Recall    :", test_results_wf.box.mr)
print("mAP50     :", test_results_wf.box.map50)
print("mAP50-95  :", test_results_wf.box.map)

print("\nPer-class mAP50-95 (test split):")
for i, name in test_results_wf.names.items():
    print(f"  {name:15s}: {test_results_wf.box.maps[i]:.3f}")

Ultralytics 8.4.121 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n-weighted-neck summary (fused): 118 layers, 2,292,756 parameters, 0 gradients, 5.2 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 1.7±2.7 ms, read: 5.5±12.3 MB/s, size: 23.8 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning /content/drive/MyDrive/Thesis/uppdms_baseline1/test/labels... 111 images, 0 backgrounds, 0 corrupt: 38% ━━━━╸─────── 111/287 3.2it/s 36.6s<55.2srequirements: Ultralytics requirement ['pi-heif'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 244ms
Prepared 1 package in 66ms
Installed 1 package in 1ms
 + pi-heif==1.4.0

requirements: AutoUpdate success ✅ 0.7s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

val: Scanning /content/drive/MyDrive/Thesis/uppdms_baseline1/test/lab

In [ ]:
baseline_model = YOLO("/content/drive/MyDrive/Thesis/Experiments1/YOLO26_Baseline/weights/best.pt")
baseline_metrics = baseline_model.val(data="/content/drive/MyDrive/Thesis/uppdms_baseline1/data.yaml", split="val")
print(baseline_metrics.box.maps)          # per-class mAP50-95
print(baseline_metrics.names)              # class name mapping

Ultralytics 8.4.121 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 122 layers, 2,376,396 parameters, 0 gradients, 5.3 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.4±0.2 ms, read: 25.9±18.8 MB/s, size: 22.4 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning /content/drive/MyDrive/Thesis/uppdms_baseline1/valid/labels.cache... 287 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 287/287 12.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.4it/s 5.4s
                   all        287       1239       0.89      0.835      0.897      0.703
            bottle_cap        119        180      0.945      0.872      0.925      0.719
                 glove         67         71      0.941      0.902      0.973      0.831
                  mask        131       

In [ ]:
from ultralytics import YOLO
import yaml

YAML_PATH = "/content/drive/MyDrive/Thesis/uppdms_baseline1/data.yaml"
best_model_path_yolo26 = "/content/drive/MyDrive/Thesis/Experiments1/YOLO26_Baseline/weights/best.pt"

# quick check on what split(s) data.yaml actually defines
with open(YAML_PATH) as f:
    print(yaml.safe_load(f))

yolo26_best = YOLO(best_model_path_yolo26)

test_results_yolo26 = yolo26_best.val(
    data=YAML_PATH,
    split="test",
    imgsz=640,
    batch=16,
    device=0,
    project="/content/drive/MyDrive/Thesis/Experiments1",
    name="YOLO26_Baseline_perclass_test_check",
)

print("\nYOLO26 BASELINE - TEST RESULTS")
print("===============================")
print("Precision :", test_results_yolo26.box.mp)
print("Recall    :", test_results_yolo26.box.mr)
print("mAP50     :", test_results_yolo26.box.map50)
print("mAP50-95  :", test_results_yolo26.box.map)

print("\nPer-class mAP50-95 (test split):")
for i, name in test_results_yolo26.names.items():
    print(f"  {name:15s}: {test_results_yolo26.box.maps[i]:.3f}")

{'train': '../train/images', 'val': '../valid/images', 'test': '../test/images', 'nc': 8, 'names': ['bottle_cap', 'glove', 'mask', 'net', 'plastic_bag', 'plastic_bottle', 'plastic_cup', 'plastic_straw'], 'roboflow': {'workspace': 'sanzida-tazin-arafa-s-workspace', 'project': 'uppdms', 'version': 2, 'license': 'CC BY 4.0', 'url': 'https://universe.roboflow.com/sanzida-tazin-arafa-s-workspace/uppdms/dataset/2'}}
Ultralytics 8.4.121 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 122 layers, 2,376,396 parameters, 0 gradients, 5.3 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.9±0.4 ms, read: 5.6±12.4 MB/s, size: 27.3 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning /content/drive/MyDrive/Thesis/uppdms_baseline1/test/labels.cache... 285 images, 0 backgrounds, 2 corrupt: 100% ━━━━━━━━━━━━ 287/287 48.2Mit/s 0.0s
val: /content/drive/My

YOLO26_Architecture

In [ ]:
from ultralytics import YOLO
import pprint

base = YOLO("yolo26n.pt")

print("========== YOLO26 YAML ==========")
pprint.pp(base.model.yaml)

========== YOLO26 YAML ==========
{'nc': 80,
 'end2end': True,
 'reg_max': 1,
 'scales': {'n': [0.5, 0.25, 1024],
            's': [0.5, 0.5, 1024],
            'm': [0.5, 1.0, 512],
            'l': [1.0, 1.0, 512],
            'x': [1.0, 1.5, 512]},
 'backbone': [[-1, 1, 'Conv', [64, 3, 2]],
              [-1, 1, 'Conv', [128, 3, 2]],
              [-1, 2, 'C3k2', [256, False, 0.25]],
              [-1, 1, 'Conv', [256, 3, 2]],
              [-1, 2, 'C3k2', [512, False, 0.25]],
              [-1, 1, 'Conv', [512, 3, 2]],
              [-1, 2, 'C3k2', [512, True]],
              [-1, 1, 'Conv', [1024, 3, 2]],
              [-1, 2, 'C3k2', [1024, True]],
              [-1, 1, 'SPPF', [1024, 5, 3, True]],
              [-1, 2, 'C2PSA', [1024]]],
 'head': [[-1, 1, 'nn.Upsample', ['None', 2, 'nearest']],
          [[-1, 6], 1, 'Concat', [1]],
          [-1, 2, 'C3k2', [512, True]],
          [-1, 1, 'nn.Upsample', ['None', 2, 'nearest']],
          [[-1, 4], 1, 'Concat', [1]],
          [

In [ ]:
print("\n========== YOLO26 LAYERS ==========")

for i, layer in enumerate(base.model.model):
    print(
        f"{i:02d}: "
        f"{type(layer).__name__}"
    )


========== YOLO26 LAYERS ==========
00: Conv
01: Conv
02: C3k2
03: Conv
04: C3k2
05: Conv
06: C3k2
07: Conv
08: C3k2
09: SPPF
10: C2PSA
11: Upsample
12: Concat
13: C3k2
14: Upsample
15: Concat
16: C3k2
17: Conv
18: Concat
19: C3k2
20: Conv
21: Concat
22: C3k2
23: Detect
